# SOKE Demo: Thai Text → Sign Language Video

Interactive wrapper around `mGPT/inference.py:run_inference`. Pick a Thai sentence from the dataset, see the retrieved KWS keywords, and get the rendered MP4 inline.

**Run order:** Cells 1–4 once per kernel (slow — model load), then Cell 5 / 6 to build the UI. Re-run the UI cell only if you want a fresh widget.

**Env:** `conda activate pytorchmacos`. If `ipywidgets` is missing: `pip install ipywidgets`.

In [1]:
# Cell 1 — Imports & headless GL setup
import os
# Must be set BEFORE pyrender / torch import (matches test.py).
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import gzip
import io
import json
import pickle
import contextlib
from pathlib import Path

import pytorch_lightning as pl
import ipywidgets as widgets
from IPython.display import Video, display, HTML, clear_output

from mGPT.config import parse_args
from mGPT.data.build_data import build_data
from mGPT.models.build_model import build_model
from mGPT.utils.logger import create_logger
from mGPT.utils.load_checkpoint import load_pretrained, load_pretrained_vae
from mGPT.inference import run_inference

REPO_ROOT = Path.cwd()
print(f"Repo root: {REPO_ROOT}")

Repo root: /mnt/hdd1/users/cham_00102/01_SOKE/SOKE


In [2]:
# Cell 2 — Load config
# parse_args() reads sys.argv, so we stuff it before calling. This reuses the
# project's full merge logic (assets + default + cfg + module configs).
CFG_PATH = "configs/soke-09-from-scratch-Thai-With-Thai-KWS.yaml"
OUTPUT_DIR = "./results/notebook_demo"
FPS = 20

_orig_argv = sys.argv
sys.argv = ["demo_infer", "--cfg", CFG_PATH, "--infer", "--src", "thai",
            "--output_dir", OUTPUT_DIR, "--fps", str(FPS)]
try:
    cfg = parse_args(phase="test")
finally:
    sys.argv = _orig_argv

os.environ["CUDA_VISIBLE_DEVICES"] = cfg.USE_GPUS
cfg.FOLDER = cfg.TEST.FOLDER
cfg.INFER_TEXT = []     # filled per submission
cfg.INFER_NAME = []     # video_id used for KWS lookup (notebook-only field)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Config loaded. Dataset: {cfg.DATASET.H2S.get('DATASET_NAME', '?')}, output: {OUTPUT_DIR}")

Force no debugging when testing
Config loaded. Dataset: thai, output: ./results/notebook_demo


In [3]:
# Cell 3 — Build model + load checkpoints (slow; run once per kernel)
logger = create_logger(cfg, phase="test")
pl.seed_everything(cfg.SEED_VALUE)

datamodule = build_data(cfg)
model = build_model(cfg, datamodule)

if cfg.TRAIN.PRETRAINED_VAE:
    load_pretrained_vae(cfg, model, logger)

if not cfg.TEST.CHECKPOINTS:
    ckpt_folder = os.path.join(cfg.FOLDER_EXP.replace("results", "experiments"), "checkpoints")
    cfg.TEST.CHECKPOINTS = os.path.join(ckpt_folder, "last.ckpt")
load_pretrained(cfg, model, logger, phase="test")
model.eval()
print(f"Model loaded from {cfg.TEST.CHECKPOINTS}")

Seed set to 1234


mean path ../data/CSL-Daily/mean.pt std_path:  ../data/CSL-Daily/std.pt


The tokenizer you are loading from './deps/mbart-h2s-csl-phoenix' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from './deps/mbart-h2s-csl-phoenix' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/519 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
2026-05-11 01:27:49,771 Loading pretrain vae from ./experiments/mgpt/DETO-Train-ALL-Thai-Dataset-Smoothed-SMPLer-X-WiLOR/checkpoints/last.ckpt
2026-05-11 01:27:49,789 Loading pretrain model from ./experiments/mgpt/SOKE-v4-Thai-With-Thai-KWS-4GPU-BS1/checkpoints/last.ckpt


load rhand vae...

=======Check Weights Loading======
Weights not used from pretrained file:
---------------------------
Weights not loaded into new model:

load hand vae...

=======Check Weights Loading======
Weights not used from pretrained file:
---------------------------
Weights not loaded into new model:

load vae...

=======Check Weights Loading======
Weights not used from pretrained file:
---------------------------
Weights not loaded into new model:

Model loaded from ./experiments/mgpt/SOKE-v4-Thai-With-Thai-KWS-4GPU-BS1/checkpoints/last.ckpt


In [4]:
# Cell 4 — Build text → video_id lookup + load name → keywords map
# Mirrors the dataset loader in mGPT/data/humanml/dataset_t2m.py:145-151
SPLIT = "train"   # train has the most coverage

thai_root = Path(cfg.DATASET.H2S.THAI_ROOT)
ann_path = thai_root / ("val_vid.train" if SPLIT == "train" else f"val_vid.{SPLIT}")
kws_template = cfg.KWS.NAME2KWS_PATH if hasattr(cfg, "KWS") else "scripts/thai_name2kws_{split}_smoothed_smplerx-wilor.json"
kws_path = Path(kws_template.format(split=SPLIT))

if not ann_path.exists():
    raise FileNotFoundError(
        f"Dataset annotation file not found: {ann_path}\n"
        f"Check cfg.DATASET.H2S.THAI_ROOT and that the dataset is mounted on this machine."
    )
if not kws_path.exists():
    raise FileNotFoundError(f"KWS file not found: {kws_path}")

with gzip.open(ann_path, "rb") as f:
    rows = pickle.load(f)
with open(kws_path, encoding="utf-8") as f:
    name2kws = json.load(f)

# Multiple videos can share the same text; keep the first occurrence.
text_to_name = {}
for r in rows:
    text_to_name.setdefault(r["text"], r["name"])

print(f"Loaded {len(rows)} samples from {ann_path.name} → {len(text_to_name)} unique texts")
print(f"Loaded {len(name2kws)} name→kws entries from {kws_path.name}")

Loaded 119 samples from val_vid.train → 119 unique texts
Loaded 119 name→kws entries from thai_name2kws_train_smoothed_smplerx-wilor.json


In [5]:
# Cell 5 — Inference helper (captures stdout/stderr so we can surface failures)
def _safe_text(text):
    # Matches the sanitization in mGPT/inference.py (line 140)
    return "".join(c if c.isalnum() or c in " -_" else "_" for c in text)[:50]

def infer_one(text, name):
    """Run a single inference. Returns (mp4_path_or_None, captured_logs_str)."""
    cfg.INFER_TEXT = [text]
    cfg.INFER_NAME = [name]
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        run_inference(cfg, model, datamodule, logger)
    logs = buf.getvalue()
    expected = Path(OUTPUT_DIR) / f"infer_0_{_safe_text(text)}.mp4"
    if expected.exists():
        return expected, logs
    # Fallback: take the most recently written mp4 in the output dir
    candidates = sorted(Path(OUTPUT_DIR).glob("*.mp4"), key=lambda p: p.stat().st_mtime)
    return (candidates[-1] if candidates else None), logs

In [8]:
# Cell 6 — UI
_options = sorted(text_to_name.keys())

text_input = widgets.Combobox(
    placeholder="พิมพ์หรือเลือกประโยคภาษาไทยจากชุดข้อมูล",
    options=_options,
    ensure_option=False,
    description="Text:",
    layout=widgets.Layout(width="700px"),
)
submit_btn = widgets.Button(description="Run inference", button_style="primary")
progress = widgets.IntProgress(value=0, min=0, max=3, description="Idle:",
                               layout=widgets.Layout(width="700px"))
out = widgets.Output()

def _render_kws(kws):
    if not kws:
        return HTML("<i>No keywords retrieved for this video_id.</i>")
    chips = "".join(
        f"<span style='display:inline-block;padding:4px 10px;margin:3px;"
        f"background:#1f77b4;color:white;border-radius:12px;font-size:14px;'>{kw}</span>"
        for kw in kws
    )
    return HTML(f"<b>Keywords retrieved:</b><br>{chips}")

def _show_logs(logs, header="Inference logs"):
    if not logs.strip():
        return
    display(HTML(f"<details open><summary><b>{header}</b></summary>"
                 f"<pre style='max-height:400px;overflow:auto;background:#111;color:#eee;"
                 f"padding:8px;font-size:12px;'>{logs}</pre></details>"))

def _on_click(_btn):
    submit_btn.disabled = True
    progress.value = 0
    progress.description = "Lookup:"
    with out:
        clear_output(wait=True)
    text = text_input.value.strip()
    try:
        if text not in text_to_name:
            with out:
                display(HTML(
                    f"<span style='color:#c00'>Text not found in dataset split <code>{SPLIT}</code>. "
                    "Pick from the autocomplete suggestions.</span>"
                ))
            return
        name = text_to_name[text]
        kws = name2kws.get(name, [])
        with out:
            # display(HTML(f"<b>Video ID:</b> <code>{name}</code>"))
            display(_render_kws(kws))
        progress.value = 1
        progress.description = "Inferring:"
        video_path, logs = infer_one(text, name)
        progress.value = 2
        progress.description = "Rendering:"
        if video_path is None or not Path(video_path).exists():
            with out:
                display(HTML(
                    "<span style='color:#c00'>Inference finished but no MP4 was produced. "
                    "See captured logs below.</span>"
                ))
                _show_logs(logs)
                existing = sorted(Path(OUTPUT_DIR).iterdir()) if Path(OUTPUT_DIR).exists() else []
                display(HTML(f"<b>Files in {OUTPUT_DIR}:</b> {[p.name for p in existing] or '(empty)'}"))
            return
        with out:
            display(HTML(f"<b>Video:</b> <code>{video_path}</code>"))
            display(Video(str(video_path), embed=True, width=512))
            _show_logs(logs)
        progress.value = 3
        progress.description = "Done:"
    except Exception as e:
        with out:
            display(HTML(f"<span style='color:#c00'>Error: {type(e).__name__}: {e}</span>"))
            import traceback
            display(HTML(f"<pre style='background:#111;color:#eee;padding:8px;font-size:12px;'>{traceback.format_exc()}</pre>"))
    finally:
        submit_btn.disabled = False

submit_btn.on_click(_on_click)
display(widgets.VBox([text_input, submit_btn, progress, out]))